In [17]:
import numpy as np
import pandas as pd

df=pd.read_csv('Life expectancy.csv')

In [18]:
df.head()

,Entity,Year,Life expectancy
0,Australia,1802,34.049999
1,Australia,1803,34.049999
2,Australia,1804,34.049999
3,Australia,1805,34.049999
4,Australia,1806,34.049999


## CatBoost

In [25]:
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import time

# Features and target
X = df.drop(columns='Life expectancy')
y = df['Life expectancy']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



# Mark categorical features
cat_features = ['Entity']

# Create Pools
pool_train = Pool(X_train, y_train, cat_features=cat_features)
pool_test = Pool(X_test, cat_features=cat_features)

# Start timer
start = time.time()

# Train CatBoost
model = CatBoostRegressor(iterations=500, learning_rate=0.1, depth=6, verbose=0)
model.fit(pool_train, eval_set=pool_test)

# Predictions
y_pred = model.predict(X_test)

# Evaluate
r2 = r2_score(y_test, y_pred)
print("R² score:", r2)

# Execution time
end = time.time()
print("Execution time:", end - start, "seconds")


R² score: 0.9861984652806269
Execution time: 10.268056631088257 seconds


## XGBoost

In [28]:
import xgboost as xgb
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import time

# Features and target
X = df.drop(columns='Life expectancy')
y = df['Life expectancy']

# Label encoding for categorical columns
lbl_entity = preprocessing.LabelEncoder()
lbl_year = preprocessing.LabelEncoder()

X['Country'] = lbl_entity.fit_transform(X['Entity'].astype(str))
X['Year'] = lbl_year.fit_transform(X['Year'].astype(str))

# Drop the original Entity column (object dtype)
X = X.drop(columns=['Entity'])

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=1
)

# Initialize and train the XGBoost Regressor
start = time.time()
xgbr = xgb.XGBRegressor()
xgbr.fit(X_train, y_train)

# Make predictions on the test set
y_pred = xgbr.predict(X_test)

# Evaluation
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("R² score:", r2)

# Execution time
end = time.time()
print("Execution time:", end - start, "seconds")


R² score: 0.9922888162201562
Execution time: 0.06668519973754883 seconds


## LightGBM

In [33]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import time

# Features and target
X = df.drop(columns='Life expectancy')
y = df['Life expectancy']

# Convert categorical columns
X['Entity'] = X['Entity'].astype('category')
X['Year'] = X['Year'].astype('category')

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=1
)

# Initialize model
lgbm = lgb.LGBMRegressor(
    boosting_type='gbdt',
    objective='regression',
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31
)

# Train model (LightGBM >=4.0 syntax)
start = time.time()
lgbm.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric='rmse',
    categorical_feature=['Entity', 'Year'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
)

# Predict
y_pred = lgbm.predict(X_test)

# Evaluation
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("R² score:", r2)
print("Execution time:", time.time() - start, "seconds")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000039 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 233
[LightGBM] [Info] Number of data points in the train set: 2602, number of used features: 2
[LightGBM] [Info] Start training from score 48.457861
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf